Block 1 — Imports and Configuration

In [14]:
import os
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from torchvision import datasets, transforms
import timm

from PIL import Image
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [ ]:

DATA_ROOT = "../../data/Leaf/RiceLeafsDisease"

TRAIN_DIR = os.path.join(DATA_ROOT, "train_aug")
VAL_DIR   = os.path.join(DATA_ROOT, "validation")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

OUT_DIR = "./outputs_vit_final"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Device & seed
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)


In [81]:
NUM_CLASSES = 6
IMG_SIZE = 224
BATCH_SIZE = 12
EPOCHS = 10
NUM_WORKERS = 2
ACCUM_STEPS = 1

# Hyperparameter sweep (FINAL)
LR_LIST = [1e-4, 5e-5, 3e-5]
WEIGHT_DECAY_LIST = [1e-4, 5e-4]
LABEL_SMOOTHING_LIST = [0.0, 0.1]
DROPOUT_LIST = [0.0, 0.2]


In [82]:
def get_transforms(img_size, train=True):
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.2, 0.2, 0.15, 0.02),
            transforms.ToTensor(),
            transforms.Normalize(
                [0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225]
            ),
        ])
    else:
        return transforms.Compose([
            transforms.Resize(int(img_size * 1.14)),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(
                [0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225]
            ),
        ])


In [83]:
def load_dataloaders():
    train_ds = datasets.ImageFolder(
        TRAIN_DIR, transform=get_transforms(IMG_SIZE, train=True)
    )
    val_ds = datasets.ImageFolder(
        VAL_DIR, transform=get_transforms(IMG_SIZE, train=False)
    )
    test_ds = datasets.ImageFolder(
        TEST_DIR, transform=get_transforms(IMG_SIZE, train=False)
    )

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE,
        shuffle=True, num_workers=NUM_WORKERS
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=NUM_WORKERS
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE,
        shuffle=False, num_workers=NUM_WORKERS
    )

    print("Classes:", train_ds.classes)
    return train_loader, val_loader, test_loader, train_ds.classes


In [84]:
class UnlabeledDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.paths = sorted([
            p for p in Path(root_dir).iterdir()
            if p.suffix.lower() in [".jpg", ".png", ".jpeg"]
        ])
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, str(self.paths[idx])


In [85]:
def build_model(num_classes, dropout):
    model = timm.create_model(
        "vit_tiny_patch16_224",
        pretrained=True,
        num_classes=num_classes,
        drop_rate=dropout
    )
    return model


In [86]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum, correct, total = 0, 0, 0
    optimizer.zero_grad()

    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels) / ACCUM_STEPS

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        loss_sum += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return loss_sum / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0, 0, 0

    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss_sum += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return loss_sum / len(loader), correct / total


In [87]:
train_loader, val_loader, test_loader, class_names = load_dataloaders()


Classes: ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scald', 'narrow_brown_spot']


In [21]:
train_loader, val_loader, test_loader, class_names = load_dataloaders()

results = []
BEST_MODEL_PATH = None
BEST_VAL_ACC = 0.0

for lr in LR_LIST:
    for wd in WEIGHT_DECAY_LIST:
        for ls in LABEL_SMOOTHING_LIST:
            for dr in DROPOUT_LIST:

                print(f"\n==== ViT | LR={lr} WD={wd} LS={ls} DR={dr} ====")

                model = build_model(NUM_CLASSES, dr).to(DEVICE)

                optimizer = optim.AdamW(
                    model.parameters(), lr=lr, weight_decay=wd
                )
                criterion = nn.CrossEntropyLoss(label_smoothing=ls)
                scaler = torch.cuda.amp.GradScaler()

                best_val_acc = 0.0

                for epoch in range(EPOCHS):
                    train_loss, train_acc = train_one_epoch(
                        model, train_loader, criterion, optimizer, scaler
                    )
                    val_loss, val_acc = evaluate(
                        model, val_loader, criterion
                    )

                    print(
                        f"Epoch {epoch+1}: "
                        f"Train Acc={train_acc*100:.2f}% | "
                        f"Val Acc={val_acc*100:.2f}%"
                    )

                    if val_acc > best_val_acc:
                        best_val_acc = val_acc
                        ckpt = f"vit_best.pt"
                        torch.save(model.state_dict(), os.path.join(OUT_DIR, ckpt))

                        if val_acc > BEST_VAL_ACC:
                            BEST_VAL_ACC = val_acc
                            BEST_MODEL_PATH = os.path.join(OUT_DIR, ckpt)

                results.append({
                    "lr": lr,
                    "weight_decay": wd,
                    "label_smoothing": ls,
                    "dropout": dr,
                    "best_val_acc": best_val_acc
                })

                del model
                torch.cuda.empty_cache()


Classes: ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scald', 'narrow_brown_spot']

==== ViT | LR=0.0001 WD=0.0001 LS=0.0 DR=0.0 ====


C:\Users\Acer\AppData\Local\Temp\ipykernel_1848\4155429764.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
  0%|          | 0/1250 [00:00<?, ?it/s]C:\Users\Acer\AppData\Local\Temp\ipykernel_1848\4269703485.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1: Train Acc=90.37% | Val Acc=95.83%


Epoch 2: Train Acc=95.83% | Val Acc=97.35%


Epoch 3: Train Acc=96.54% | Val Acc=96.97%


...

Epoch 6: Train Acc=98.82% | Val Acc=97.35%


Epoch 7: Train Acc=98.71% | Val Acc=97.73%


Epoch 8: Train Acc=99.02% | Val Acc=98.86%


Epoch 9: Train Acc=99.22% | Val Acc=98.48%


Epoch 10: Train Acc=99.02% | Val Acc=98.48%


| Model | Learning Rate | Weight Decay | Label Smoothing | Dropout | Best Train Acc (%) | Best Val Acc (%) | Epoch (Best Val) |
|------|---------------|--------------|-----------------|---------|-------------------|------------------|------------------|
| ViT  | 1e-4  | 1e-4 | 0.0 | 0.0 | 98.43 | 97.35 | 2 |
| ViT  | 1e-4  | 1e-4 | 0.0 | 0.2 | 98.15 | 98.48 | 10 |
| ViT  | 1e-4  | 1e-4 | 0.1 | 0.0 | 98.38 | 98.48 | 10 |
| ViT  | 1e-4  | 1e-4 | 0.1 | 0.2 | 98.18 | 97.73 | 4 |
| ViT  | 1e-4  | 5e-4 | 0.0 | 0.0 | 98.32 | 97.35 | 10 |
| ViT  | 1e-4  | 5e-4 | 0.0 | 0.2 | 98.14 | 98.86 | 8 |
| ViT  | 1e-4  | 5e-4 | 0.1 | 0.0 | 98.38 | 98.48 | 4 |
| ViT  | 1e-4  | 5e-4 | 0.1 | 0.2 | 98.28 | 98.86 | 3 |
| ViT  | 5e-5  | 1e-4 | 0.0 | 0.0 | 98.73 | 98.86 | 10 |
| ViT  | 5e-5  | 1e-4 | 0.0 | 0.2 | 98.18 | 98.48 | 2 |
| ViT  | 5e-5  | 1e-4 | 0.1 | 0.0 | 98.43 | 98.11 | 4 |
| ViT  | 5e-5  | 1e-4 | 0.1 | 0.2 | 98.25 | 98.48 | 5 |
| ViT  | 5e-5  | 5e-4 | 0.0 | 0.0 | 98.00 | 98.48 | 4 |
| ViT  | 5e-5  | 5e-4 | 0.0 | 0.2 | 98.46 | 98.86 | 7 |
| ViT  | 5e-5  | 5e-4 | 0.1 | 0.0 | 98.67 | 99.24 | 6 |
| ViT  | 5e-5  | 5e-4 | 0.1 | 0.2 | 98.61 | 97.73 | 7 |
| ViT  | 3e-5  | 1e-4 | 0.0 | 0.0 | 99.25 | 98.48 | 9 |
| ViT  | 3e-5  | 1e-4 | 0.0 | 0.2 | 98.87 | 98.11 | 10 |
| ViT  | 3e-5  | 1e-4 | 0.1 | 0.0 | 99.10 | 98.48 | 4 |
| ViT  | 3e-5  | 1e-4 | 0.1 | 0.2 | 99.11 | 98.48 | 5 |
| ViT  | 3e-5  | 5e-4 | 0.0 | 0.0 | 99.24 | 98.48 | 4 |
| ViT  | 3e-5  | 5e-4 | 0.0 | 0.2 | 98.35 | 98.11 | 5 |
| ViT  | 3e-5  | 5e-4 | 0.1 | 0.0 | 99.11 | **99.24** | 8 |
| ViT  | 3e-5  | 5e-4 | 0.1 | 0.2 | 99.22 | 98.86 | 8 |


The best model is saved based on the highest validation accuracy with Learning rate 3e-5, weight decay 5e-3, label smooting 0.1, dropout 0 with the best validation accuracy of 99.24 at epoch 8.